# AAROH — Audio Emotion Transformer Training (Slice 3.4)

This notebook trains the **Audio Emotion Representation Model** using HuggingFace `facebook/wav2vec2-base` on RAVDESS.

## Strict Boundaries
- Audio Emotion != Clinical Distress
- Does NOT predict distress_score, escalation_probability, depression, anxiety, risk level, or diagnosis
- Preserves Voice Service boundary (ASR, VAD, pause ratio, etc.)

## Steps
1. Clone repository & install dependencies
2. Verify CUDA GPU
3. Run smoke test (rapid end-to-end verification)
4. Run full HuggingFace transformer training
5. Export production artifacts
6. Verify model loads

## 1. Clone Repository & Install Dependencies

In [ ]:
# Clone the AAROH repository
!git clone https://github.com/Diya579/AAROH.git
%cd AAROH

# Install dependencies
!pip install -q torch torchaudio transformers datasets librosa scikit-learn numpy

## 2. Verify CUDA GPU

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("WARNING: No GPU detected. Enable GPU runtime: Runtime -> Change runtime type -> T4 GPU")

## 3. Run Smoke Test (Rapid End-to-End Verification)

In [ ]:
import sys
sys.path.insert(0, '.')

from backend.ml.training.train_audio_emotion import parse_args, train_audio_emotion

# Run smoke test with FP16 on CUDA
smoke_args = parse_args([
    "--smoke-test",
    "--fp16",
    "--execution-mode", "PYTORCH_FROZEN",
    "--output-dir", "checkpoints/audio_emotion/colab_smoke_artifact",
    "--checkpoint-dir", "checkpoints/audio_emotion/colab_smoke_ckpt",
    "--gradient-accumulation-steps", "2",
    "--warmup-ratio", "0.10",
    "--weight-decay", "0.01",
])

smoke_report = train_audio_emotion(smoke_args)

# Assertions
assert smoke_report["forward_pass_successful"], "Forward pass failed!"
assert smoke_report["backward_pass_successful"], "Backward pass failed!"
assert smoke_report["optimizer_step_successful"], "Optimizer step failed!"
assert smoke_report["checkpoint_saved"], "Checkpoint not saved!"
assert smoke_report["checkpoint_reloaded"], "Checkpoint reload failed!"
assert smoke_report["inference_after_reload_success"], "Inference after reload failed!"
assert smoke_report["all_exported_files_exist"], "Exported artifact files missing!"
print("\n✅ SMOKE TEST PASSED — All checks verified.")

## 4. Run Full HuggingFace Transformer Training

### Option A: Frozen Backbone (Recommended for RAVDESS)
Trains only the classification head on top of frozen wav2vec2 features.

In [ ]:
# Full training run — Frozen backbone
train_args = parse_args([
    "--execution-mode", "PYTORCH_FROZEN",
    "--model-name", "facebook/wav2vec2-base",
    "--epochs", "5",
    "--batch-size", "16",
    "--lr", "3e-4",
    "--gradient-accumulation-steps", "2",
    "--weight-decay", "0.01",
    "--warmup-ratio", "0.10",
    "--fp16",
    "--seed", "42",
    "--output-dir", "checkpoints/audio_emotion/colab_finetune_run",
    "--checkpoint-dir", "checkpoints/audio_emotion/colab_finetune_checkpoints",
    "--drive-checkpoint-dir", "/content/drive/MyDrive/aaroh_checkpoints/audio_emotion",
])

report = train_audio_emotion(train_args)
print(f"\n✅ TRAINING COMPLETE — Validation Accuracy: {report['metrics']['accuracy']:.4f}")

### Option B: Fine-Tune Backbone (Unfreeze wav2vec2)
Unfreezes the wav2vec2 encoder for full fine-tuning. Use lower backbone LR.

In [ ]:
# Quality-focused fine-tuning run — Unfrozen wav2vec2 backbone
finetune_args = parse_args([
    "--execution-mode", "PYTORCH_FINETUNE",
    "--model-name", "facebook/wav2vec2-base",
    "--unfreeze-backbone",
    "--epochs", "5",
    "--batch-size", "8",
    "--lr-transformer", "1e-5",
    "--lr-head", "1e-4",
    "--gradient-accumulation-steps", "4",
    "--weight-decay", "0.01",
    "--warmup-ratio", "0.10",
    "--early-stopping-patience", "2",
    "--eval-batch-size", "16",
    "--fp16",
    "--seed", "42",
    "--output-dir", "checkpoints/audio_emotion/colab_finetune_unfrozen_run",
    "--checkpoint-dir", "checkpoints/audio_emotion/colab_finetune_unfrozen_checkpoints",
    "--drive-checkpoint-dir", "/content/drive/MyDrive/aaroh_checkpoints/audio_emotion_unfrozen",
])

finetune_report = train_audio_emotion(finetune_args)
print(f"\n✅ FINE-TUNING COMPLETE — Validation Accuracy: {finetune_report['metrics']['accuracy']:.4f}")

## 5. Export Production Artifacts

The training script automatically exports to `models/audio_emotion/`. To promote the Colab-trained checkpoint to production:

In [ ]:
import shutil
from pathlib import Path

# Promote best Colab checkpoint to production models/audio_emotion/
colab_run_dir = Path("checkpoints/audio_emotion/colab_finetune_run")
prod_dir = Path("models/audio_emotion")

if colab_run_dir.exists():
    # Copy all exported artifacts to production
    prod_dir.mkdir(parents=True, exist_ok=True)
    for f in colab_run_dir.iterdir():
        if f.is_file():
            shutil.copy2(f, prod_dir / f.name)
    print(f"✅ Production artifacts exported to {prod_dir}")
    print("Files:")
    for f in sorted(prod_dir.iterdir()):
        print(f"  - {f.name}")
else:
    print("⚠️ Colab run directory not found. Run training first.")

## 6. Verify Model Loads

Load the exported model from disk and verify inference works.

In [ ]:
from backend.ml.training.models.audio_emotion.model import AudioEmotionModel

# Load model directly from production artifacts
model = AudioEmotionModel.load_from_artifact("models/audio_emotion", device="cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Model loaded from artifact")
print(f"  Execution Mode: {model.execution_mode}")
print(f"  Backbone: {model.backbone}")
print(f"  Trainable Params: {model.trainable_parameters_count:,}")
print(f"  Frozen Params: {model.frozen_parameters_count:,}")

# Verify inference on a sample
import json
from backend.ml.training.preprocessing.common import read_jsonl

records = read_jsonl("datasets/processed/ravdess.jsonl")
sample_path = records[0]["audio_path"]
result = model.predict_audio_embedding(sample_path)

assert "audio_embedding" in result, "Missing audio_embedding"
assert "audio_emotion_probabilities" in result, "Missing audio_emotion_probabilities"
assert len(result["audio_embedding"]) == 768, f"Expected 768-dim embedding, got {len(result['audio_embedding'])}"
assert len(result["audio_emotion_probabilities"]) == 8, f"Expected 8 emotion classes, got {len(result['audio_emotion_probabilities'])}"

print(f"\n✅ INFERENCE VERIFIED")
print(f"  Audio Embedding Dim: {len(result['audio_embedding'])}")
print(f"  Emotion Probabilities: {result['audio_emotion_probabilities']}")

## 7. Verify Full Pipeline Loading

Verify the production inference pipeline can load the audio model.

In [ ]:
from backend.ml.inference import MLInferencePipeline, PipelineConfig, ModelRegistry

# Load pipeline in FALLBACK mode (all models must be present)
config = PipelineConfig(default_execution_mode="FALLBACK", strict_stage_validation=True)
registry = ModelRegistry(base_dir="models")
pipeline = MLInferencePipeline(config=config, registry=registry)
pipeline.load_models()

health = pipeline.health_check()
print(f"Pipeline Health: {health['overall_status']}")
print(f"Audio Model Loaded: {health['audio_loaded']}")
print(f"Artifacts Valid: {health['artifacts_valid']}")
print(f"\n✅ FULL PIPELINE LOADED SUCCESSFULLY")